In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, cohen_kappa_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight


In [2]:
train= pd.read_parquet("..//preprocessed//train_final.parquet")
test= pd.read_parquet("..//preprocessed//test_final.parquet")

In [3]:
display(train.head(30))
train.info()

,Product_Info_1,Product_Info_2,Product_Info_3,Product_Info_4,Product_Info_5,Product_Info_6,Product_Info_7,Ins_Age,Ht,Wt,...,Medical_Keyword_40,Medical_Keyword_41,Medical_Keyword_42,Medical_Keyword_43,Medical_Keyword_44,Medical_Keyword_45,Medical_Keyword_46,Medical_Keyword_47,Medical_Keyword_48,Response
0,1,D3,10,0.076923,2,1,1,0.641791,0.581818,0.148536,...,0,0,0,0,0,0,0,0,0,8
1,1,A1,26,0.076923,2,3,1,0.059701,0.600000,0.131799,...,0,0,0,0,0,0,0,0,0,4
2,1,E1,26,0.076923,2,3,1,0.029851,0.745455,0.288703,...,0,0,0,0,0,0,0,0,0,8
3,1,D4,10,0.487179,2,3,1,0.164179,0.672727,0.205021,...,0,0,0,0,0,0,0,0,0,8
4,1,D2,26,0.230769,2,3,1,0.417910,0.654545,0.234310,...,0,0,0,0,0,0,0,0,0,8
5,1,D2,26,0.230769,3,1,1,0.507463,0.836364,0.299163,...,0,0,0,0,0,0,0,0,0,8
6,1,A8,10,0.166194,2,3,1,0.373134,0.581818,0.173640,...,0,0,0,0,0,0,0,0,0,8
7,1,D2,26,0.076923,2,3,1,0.611940,0.781818,0.403766,...,0,0,0,0,0,0,0,0,0,1
8,1,D3,26,0.230769,2,3,1,0.522388,0.618182,0.184100,...,0,0,0,0,0,0,0,0,0,8
9,1,E1,21,0.076923,2,3,1,0.552239,0.600000,0.284519,...,0,0,0,0,0,0,0,0,0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59381 entries, 0 to 59380
Columns: 123 entries, Product_Info_1 to Response
dtypes: category(1), float64(14), int64(108)
memory usage: 55.3 MB


Iako je izvrsena konverzija tipova, Python je kategorijske atribute prepoznao kao numericke pa ce ponovna konverzija biti potrebna.

In [4]:
display(test.head(30))
test.info()

,Product_Info_1,Product_Info_2,Product_Info_3,Product_Info_4,Product_Info_5,Product_Info_6,Product_Info_7,Ins_Age,Ht,Wt,...,Medical_Keyword_39,Medical_Keyword_40,Medical_Keyword_41,Medical_Keyword_42,Medical_Keyword_43,Medical_Keyword_44,Medical_Keyword_45,Medical_Keyword_46,Medical_Keyword_47,Medical_Keyword_48
0,1,D3,26,0.487179,2,3,1,0.611940,0.781818,0.338912,...,0,0,0,0,0,0,0,0,0,0
1,1,A2,26,0.076923,2,3,1,0.626866,0.727273,0.311715,...,0,0,0,0,0,0,0,0,0,0
2,1,D3,26,0.144667,2,3,1,0.582090,0.709091,0.320084,...,0,0,0,0,0,0,0,0,0,0
3,1,A1,26,0.151709,2,1,1,0.522388,0.654545,0.267782,...,0,0,0,0,0,0,0,0,1,1
4,1,A1,26,0.076923,2,3,1,0.298507,0.672727,0.246862,...,0,0,0,0,0,0,0,0,0,0
5,1,D3,26,0.230769,2,3,1,0.567164,0.818182,0.299163,...,0,0,0,0,0,0,0,0,0,0
6,1,A3,26,1.000000,2,3,1,0.298507,0.690909,0.225941,...,0,0,0,0,0,0,0,0,0,0
7,1,D4,26,0.256410,2,3,1,0.059701,0.654545,0.215481,...,0,0,0,0,0,0,0,0,0,0
8,1,D3,26,0.076923,2,3,1,0.611940,0.618182,0.225941,...,0,0,0,0,0,0,0,0,0,0
9,1,A3,26,0.076923,2,3,1,0.328358,0.781818,0.320084,...,0,0,0,0,0,0,0,0,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19765 entries, 0 to 19764
Columns: 122 entries, Product_Info_1 to Medical_Keyword_48
dtypes: category(1), float64(14), int64(107)
memory usage: 18.3 MB


In [5]:
target = "Response"

cat_cols = [
    "Product_Info_1","Product_Info_2","Product_Info_3","Product_Info_5","Product_Info_6","Product_Info_7",
    "Employment_Info_2","Employment_Info_3","Employment_Info_5",
    "InsuredInfo_1","InsuredInfo_2","InsuredInfo_3","InsuredInfo_4","InsuredInfo_5","InsuredInfo_6","InsuredInfo_7",
    "Insurance_History_1","Insurance_History_2","Insurance_History_3","Insurance_History_4","Insurance_History_7",
    "Insurance_History_8","Insurance_History_9","Family_Hist_1", "Medical_History_2",
    "Medical_History_3","Medical_History_4","Medical_History_5","Medical_History_6","Medical_History_7",
    "Medical_History_8","Medical_History_9","Medical_History_11","Medical_History_12","Medical_History_13","Medical_History_14",
    "Medical_History_16","Medical_History_17","Medical_History_18","Medical_History_19","Medical_History_20","Medical_History_21",
    "Medical_History_22","Medical_History_23","Medical_History_25","Medical_History_26","Medical_History_27","Medical_History_28",
    "Medical_History_29","Medical_History_30","Medical_History_31","Medical_History_33","Medical_History_34","Medical_History_35",
    "Medical_History_36","Medical_History_37","Medical_History_38","Medical_History_39","Medical_History_40","Medical_History_41"
]

continuous_cols = [
    "Product_Info_4","Ins_Age","Ht","Wt","BMI",
    "Employment_Info_1","Employment_Info_4","Employment_Info_6",
    "Insurance_History_5",
    "Family_Hist_2","Family_Hist_3","Family_Hist_4","Family_Hist_5"
]

discrete_cols = ["Medical_History_1"]

dummy_cols = [f"Medical_Keyword_{i}" for i in range(1,49)]

In [6]:
def numeric_to_categorical(df, columns):
    for col in columns:
        df[col] = df[col].astype('category')
    
    return df
train = numeric_to_categorical(train, cat_cols)
test = numeric_to_categorical(test, cat_cols)

In [7]:
for col in discrete_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

In [8]:
 train_eval, val_eval = train_test_split(
        train,
        test_size=0.2,
        stratify=train["Response"],
        random_state=42
    )

    

In [9]:
missing_percent = train_eval.isnull().mean()*100

missing_percent = missing_percent[
    missing_percent>0
].sort_values(ascending=False)

print(missing_percent)

Family_Hist_5          70.423543
Family_Hist_3          57.561469
Family_Hist_2          48.280145
Insurance_History_5    42.636410
Family_Hist_4          32.275177
Employment_Info_6      18.238464
Medical_History_1      14.929269
Employment_Info_4      11.403250
Employment_Info_1       0.033681
dtype: float64


Pored atributa koji su izbaceni iz razmatranja u ranijim koracima, i dalje postoje atributi sa visokim procentom nedostajucih vrednosti. Imajuci u vidu da imputacija predstavlja procenu nedostajucih vrednosti na osnovu postojecih podataka, veliki procenat nedostajucih vrednosti moze dovesti do gubitka pouzdanosti i potencijalnog unosenja suma u podatke. Smatra se da procenat od oko 70% nedostajucih vrednosti predstavlja znacajni nedostatak. Ipak, u ovoj fazi analize svi atributi ce biti zadrzani kako bi se ispitalo ponasanje razlicitih modela, sa i bez istih.

### Imputacija nedostajucih vrednosti

In [10]:
numeric_with_missing = [
    col for col in continuous_cols
    if train_eval[col].isnull().sum() > 0
]

print(numeric_with_missing)

['Employment_Info_1', 'Employment_Info_4', 'Employment_Info_6', 'Insurance_History_5', 'Family_Hist_2', 'Family_Hist_3', 'Family_Hist_4', 'Family_Hist_5']


In [11]:
train_eval[numeric_with_missing].skew().sort_values()

Family_Hist_5          -0.725374
Family_Hist_3          -0.565316
Family_Hist_4           0.199528
Family_Hist_2           0.212059
Employment_Info_6       0.783921
Employment_Info_1       4.733737
Employment_Info_4      14.989680
Insurance_History_5    94.440320
dtype: float64

Na osnovu analize asimetrije numerickih promenljivih uoceno je da vecina atributa odstupa od simetricne raspodele i sadrzi ekstremne vrednosti. Zbog robusnosti na prisustvo odstupajucih vrednosti, za imputaciju numerickih atributa odabrana je medijana.

In [12]:
num_imputer = SimpleImputer(strategy='median')

train_eval[numeric_with_missing] = num_imputer.fit_transform(
    train_eval[numeric_with_missing]
)

val_eval[numeric_with_missing] = num_imputer.transform(
    val_eval[numeric_with_missing]
)



In [13]:
dummy_with_missing = [
    col for col in dummy_cols
    if train_eval[col].isnull().sum() > 0
]
print(dummy_with_missing)

[]


In [14]:
discrete_with_missing = [
    col for col in discrete_cols
    if train_eval[col].isnull().sum() > 0
]
print(discrete_with_missing)

['Medical_History_1']


In [15]:
cat_imputer = SimpleImputer(
    strategy="most_frequent"
)

train_eval[discrete_with_missing] = cat_imputer.fit_transform(
    train_eval[discrete_with_missing]
)

val_eval[discrete_with_missing] = cat_imputer.transform(
    val_eval[discrete_with_missing]
)


In [16]:
cat_with_missing =  [ col for col in cat_cols
    if train_eval[col].isnull().sum() > 0
]
print(cat_with_missing)

[]


In [17]:
def overview(df):
    pd.set_option('display.max_rows', None)
    print("(Broj_redova, broj_kolona): ", df.shape)
    print("Tip podataka svake kolone:\n", df.dtypes.to_frame(name='Tip_podatka'))
    missing = pd.DataFrame({'Broj_NA': df.isna().sum(),'Procenat_NA': round(df.isna().sum()/len(df)*100,2)}).sort_values(by='Procenat_NA',ascending=False)
    print("NA vrednosti:\n", missing)
    print("Broj razlicitih vrednosti:\n", df.nunique())
    for col in discrete_cols + cat_cols:
        print(col, ":", df[col].nunique())

In [18]:
overview(train_eval)
overview(val_eval)

(Broj_redova, broj_kolona):  (47504, 123)
Tip podataka svake kolone:
                     Tip_podatka
Product_Info_1         category
Product_Info_2         category
Product_Info_3         category
Product_Info_4          float64
Product_Info_5         category
Product_Info_6         category
Product_Info_7         category
Ins_Age                 float64
Ht                      float64
Wt                      float64
BMI                     float64
Employment_Info_1       float64
Employment_Info_2      category
Employment_Info_3      category
Employment_Info_4       float64
Employment_Info_5      category
Employment_Info_6       float64
InsuredInfo_1          category
InsuredInfo_2          category
InsuredInfo_3          category
InsuredInfo_4          category
InsuredInfo_5          category
InsuredInfo_6          category
InsuredInfo_7          category
Insurance_History_1    category
Insurance_History_2    category
Insurance_History_3    category
Insurance_History_4    category
In

### Transformacija promenljivih sa velikim brojem kategorija

In [19]:
freq = train_eval['Medical_History_2'].value_counts(normalize=True)*100

print(freq.head(30))

Medical_History_2
112    18.543702
162    14.028292
491     9.649714
335     5.073257
161     5.016420
16      4.986948
261     3.900724
478     3.264988
3       2.397693
628     1.993516
565     1.684069
155     1.656703
145     1.559869
613     1.252526
366     1.189374
387     1.069384
132     0.797828
156     0.768356
434     0.724150
453     0.696783
181     0.639946
528     0.564163
276     0.559953
217     0.446278
600     0.439963
14      0.431543
202     0.425227
52      0.425227
57      0.418912
278     0.387336
Name: proportion, dtype: float64


In [20]:
rare = freq[freq < 0.5].index
train_eval['Medical_History_2'] = train_eval['Medical_History_2'].apply(
    lambda x: "Other" if x in rare else x
)

val_eval['Medical_History_2'] = val_eval['Medical_History_2'].apply(
    lambda x: "Other" if x in rare else x
)

In [21]:
print(train_eval['Medical_History_2'].nunique())
print(val_eval['Medical_History_2'].nunique())

24
24


In [22]:
print(train_eval['Medical_History_2'].unique())
print(train_eval['Medical_History_2'].dtype)
print(val_eval['Medical_History_2'].unique())
print(val_eval['Medical_History_2'].dtype)

['Other' 613 491 276 112 145 155 162 3 261 478 16 156 132 335 387 434 565
 161 528 181 628 453 366]
object
['Other' 453 491 261 161 162 565 16 145 112 3 366 181 335 478 132 387 628
 276 156 613 155 434 528]
object


In [23]:
train_eval['Medical_History_2'] = train_eval['Medical_History_2'].astype(str).astype('category')

val_eval['Medical_History_2'] = val_eval['Medical_History_2'].astype(str).astype('category')

In [24]:
freq = train_eval['Medical_History_1'].value_counts(normalize=True)*100

print(freq.head(30))

Medical_History_1
1.0     27.822920
2.0      9.691815
0.0      8.020377
3.0      7.980381
4.0      6.195268
5.0      5.319552
6.0      5.022735
7.0      3.523914
12.0     2.911334
8.0      2.795554
9.0      2.500842
10.0     2.267178
11.0     2.001937
13.0     1.458825
14.0     1.199899
15.0     0.966234
16.0     0.848350
18.0     0.772567
17.0     0.736780
24.0     0.631526
19.0     0.522061
21.0     0.479960
22.0     0.465224
23.0     0.454699
20.0     0.444173
27.0     0.349444
25.0     0.294712
26.0     0.284187
30.0     0.225244
28.0     0.216824
Name: proportion, dtype: float64


In [25]:
rare = freq[freq < 0.5].index
train_eval['Medical_History_1'] = train_eval['Medical_History_1'].apply(
    lambda x: "Other" if x in rare else x
)

val_eval['Medical_History_1'] = val_eval['Medical_History_1'].apply(
    lambda x: "Other" if x in rare else x
)

In [26]:
print(train_eval['Medical_History_1'].unique())
print(train_eval['Medical_History_1'].dtype)
print(val_eval['Medical_History_1'].unique())
print(val_eval['Medical_History_1'].dtype)

[1.0 8.0 9.0 2.0 4.0 14.0 0.0 7.0 3.0 10.0 'Other' 19.0 12.0 5.0 6.0 13.0
 11.0 17.0 15.0 24.0 18.0 16.0]
object
[3.0 0.0 5.0 9.0 12.0 7.0 1.0 'Other' 17.0 8.0 10.0 2.0 4.0 6.0 16.0 18.0
 14.0 11.0 19.0 15.0 24.0 13.0 187.0 175.0 162.0 127.0 176.0 130.0 143.0
 185.0]
object


In [27]:
train_eval['Medical_History_1'] = train_eval['Medical_History_1'].astype(str).astype('category')

val_eval['Medical_History_1'] = val_eval['Medical_History_1'].astype(str).astype('category')

In [28]:
train_eval_start = train_eval.copy()
val_eval_start = val_eval.copy()

def get_start_data():
    return train_eval_start.copy(), val_eval_start.copy()

### Logisticka regresija

In [29]:
train_eval, val_eval = get_start_data()
X_train, y_train = train_eval.drop(columns="Response"), train_eval["Response"]
X_val, y_val = val_eval.drop(columns="Response"), val_eval["Response"]


In [30]:
cols_to_encode = cat_cols + discrete_cols

preprocessor = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", drop="first"),
            cols_to_encode
        )
    ],
    remainder="passthrough"
)



log_reg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=300, C=1,
    penalty='l2',
     random_state=42,
    class_weight="balanced"))
])

In [31]:
log_reg_pipeline.fit(X_train, y_train)


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [32]:
def evaluate_model(model, X_val, y_val):
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)

    print("Accuracy:", round(accuracy_score(y_val, y_pred), 4))

    print("Quadratic Kappa:",
          round(cohen_kappa_score(y_val, y_pred, weights="quadratic"), 4))

    print("AUC:",
          round(roc_auc_score(y_val, y_proba, multi_class="ovr", average="weighted"), 4))

    print("\nClassification report:")
    print(classification_report(y_val, y_pred))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_val, y_pred))

In [33]:
evaluate_model(log_reg_pipeline, X_val, y_val)


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.4322
Quadratic Kappa: 0.5003
AUC: 0.794

Classification report:
              precision    recall  f1-score   support

           1       0.35      0.33      0.34      1241
           2       0.32      0.23      0.27      1310
           3       0.10      0.43      0.16       203
           4       0.13      0.59      0.21       286
           5       0.35      0.40      0.38      1086
           6       0.42      0.16      0.24      2247
           7       0.39      0.38      0.39      1606
           8       0.73      0.70      0.71      3898

    accuracy                           0.43     11877
   macro avg       0.35      0.41      0.34     11877
weighted avg       0.48      0.43      0.44     11877


Confusion matrix:
[[ 411  186   96   99  117   82  102  148]
 [ 235  307  155   95  233   64   93  128]
 [   9   18   88   51   20    8    2    7]
 [   8    2   59  169    0   20    3   25]
 [  79  132  144   39  439   57  104   92]
 [ 229  188  253  375  242  370  303  2

In [34]:
param_grid = {
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"],
        "model__C": [0.01, 0.1, 1, 10]
    }

grid_log_reg = GridSearchCV(
    log_reg_pipeline,
    param_grid,
    cv=3,
    scoring=make_scorer(
        cohen_kappa_score,
        weights="quadratic"
    )
)

grid_log_reg.fit(X_train, y_train)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 33, 50, 59] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 5, 6, 18, 36, 40, 43, 55] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 37] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You m

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__C': [0.01, 0.1, ...], 'model__penalty': ['l2'], 'model__solver': ['lbfgs']}"
,scoring,make_scorer(c...hts=quadratic)
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehot', ...)]"


In [35]:
print(grid_log_reg.best_params_)
print(grid_log_reg.best_score_)

{'model__C': 1, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
0.49598937240375784


In [36]:
best_log_reg = grid_log_reg.best_estimator_

evaluate_model(
    best_log_reg,
    X_val,
    y_val
)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.4322
Quadratic Kappa: 0.5003
AUC: 0.794

Classification report:
              precision    recall  f1-score   support

           1       0.35      0.33      0.34      1241
           2       0.32      0.23      0.27      1310
           3       0.10      0.43      0.16       203
           4       0.13      0.59      0.21       286
           5       0.35      0.40      0.38      1086
           6       0.42      0.16      0.24      2247
           7       0.39      0.38      0.39      1606
           8       0.73      0.70      0.71      3898

    accuracy                           0.43     11877
   macro avg       0.35      0.41      0.34     11877
weighted avg       0.48      0.43      0.44     11877


Confusion matrix:
[[ 411  186   96   99  117   82  102  148]
 [ 235  307  155   95  233   64   93  128]
 [   9   18   88   51   20    8    2    7]
 [   8    2   59  169    0   20    3   25]
 [  79  132  144   39  439   57  104   92]
 [ 229  188  253  375  242  370  303  2

In [37]:
evaluate_model(
    best_log_reg,
    X_train,
    y_train
)

Accuracy: 0.4443
Quadratic Kappa: 0.5072
AUC: 0.804

Classification report:
              precision    recall  f1-score   support

           1       0.35      0.32      0.34      4966
           2       0.34      0.26      0.30      5242
           3       0.12      0.58      0.20       810
           4       0.15      0.67      0.25      1142
           5       0.38      0.43      0.40      4346
           6       0.41      0.17      0.24      8986
           7       0.38      0.38      0.38      6421
           8       0.73      0.71      0.72     15591

    accuracy                           0.44     47504
   macro avg       0.36      0.44      0.35     47504
weighted avg       0.48      0.44      0.45     47504


Confusion matrix:
[[ 1603   703   461   335   528   285   494   557]
 [  893  1379   553   321   798   326   462   510]
 [   32    41   466   144    47    43    15    22]
 [   22    12   162   770    14    48    27    87]
 [  338   508   608   148  1853   251   340   300]

Logisticka regresija ostvarila je Accuracy vrednost od 0.4443, Quadratic Kappa vrednost od 0.5072 i AUC od 0.804. Iako Accuracy nije izrazito visoka, Kappa vrednost ukazuje na umereno slaganje predikcija modela sa stvarnim klasama, dok relativno visoka AUC vrednost pokazuje da model poseduje sposobnost razlikovanja klasa. Nije uocena znacajna razlika izmedju performansi na trening i validacionom skupu, sto ukazuje da nije doslo do izrazenog overfittinga i da model relativno dobro generalizuje na nevidjene podatke.

Analizom klasifikacionog izvestaja primeceno je da model ima izrazite poteskoce pri klasifikaciji manje zastupljenih klasa. Posebno su niske precision vrednosti uocene za klase 3 i 4, sto ukazuje da model cesto pogresno dodeljuje opservacije ovim klasama. Istovremeno, recall za iste klase je relativno visok, sto znaci da model uspeva da prepozna veci broj stvarnih pripadnika ovih klasa, ali uz znacajan broj lazno pozitivnih klasifikacija.

Sa druge strane, klasa 8, koja predstavlja najzastupljeniju klasu u skupu podataka, ostvarila je najbolje rezultate po svim metrikama (precision=0.73, recall=0.71), sto ukazuje da model favorizuje cesce klase i lakse uci njihove obrasce. Ovakvo ponasanje moze ukazivati na uticaj nebalansiranosti klasa i predstavlja potencijalni prostor za unapredjenje kroz dodatne tehnike balansiranja ili modifikacije skupa atributa u narednim iteracijama.

### Decision tree

In [38]:
tree_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

In [39]:
tree_pipeline.fit(X_train, y_train)
evaluate_model(tree_pipeline, X_val, y_val)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.4007
Quadratic Kappa: 0.3588
AUC: 0.6376

Classification report:
              precision    recall  f1-score   support

           1       0.21      0.20      0.21      1241
           2       0.24      0.23      0.23      1310
           3       0.14      0.16      0.15       203
           4       0.16      0.17      0.17       286
           5       0.38      0.41      0.39      1086
           6       0.34      0.33      0.33      2247
           7       0.30      0.32      0.31      1606
           8       0.65      0.62      0.63      3898

    accuracy                           0.40     11877
   macro avg       0.30      0.31      0.30     11877
weighted avg       0.41      0.40      0.40     11877


Confusion matrix:
[[ 254  225   33   23  145  200  153  208]
 [ 208  298   32   27  189  225  152  179]
 [  30   30   33    7   31   42   12   18]
 [  27   22   15   50    8   77   23   64]
 [ 125  154   42    6  441  178   64   76]
 [ 219  204   45   75  207  743  364  

In [40]:
param_grid = {
   "model__max_depth": [10, 20, 30],
    "model__min_samples_split": [50, 100, 300],
    "model__min_samples_leaf": [20, 50, 100]
}

In [41]:
grid_tree = GridSearchCV(
    tree_pipeline,
    param_grid,
    cv=3,
    verbose=2,
    n_jobs=1,
    scoring= make_scorer(
        cohen_kappa_score,
        weights="quadratic"
    )
)

grid_tree.fit(X_train, y_train)

Fitting 3 folds for each of 27 candidates, totalling 81 fits


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 33, 50, 59] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__max_depth=10, model__min_samples_leaf=20, model__min_samples_split=50; total time=   5.2s


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 5, 6, 18, 36, 40, 43, 55] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__max_depth=10, model__min_samples_leaf=20, model__min_samples_split=50; total time=   5.2s


...

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 5, 6, 18, 36, 40, 43, 55] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__max_depth=30, model__min_samples_leaf=100, model__min_samples_split=300; total time=   5.0s


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 37] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__max_depth=30, model__min_samples_leaf=100, model__min_samples_split=300; total time=   5.3s


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [10, 20, ...], 'model__min_samples_leaf': [20, 50, ...], 'model__min_samples_split': [50, 100, ...]}"
,scoring,make_scorer(c...hts=quadratic)
,n_jobs,1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehot', ...)]"


In [42]:
print(grid_tree.best_params_)
print(grid_tree.best_score_)

{'model__max_depth': 10, 'model__min_samples_leaf': 20, 'model__min_samples_split': 300}
0.4397852059476588


In [43]:
best_tree = grid_tree.best_estimator_

evaluate_model(
    best_tree,
    X_val,
    y_val
)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.4274
Quadratic Kappa: 0.4252
AUC: 0.7855

Classification report:
              precision    recall  f1-score   support

           1       0.31      0.15      0.20      1241
           2       0.33      0.17      0.23      1310
           3       0.11      0.49      0.18       203
           4       0.11      0.67      0.18       286
           5       0.51      0.50      0.51      1086
           6       0.40      0.15      0.21      2247
           7       0.37      0.51      0.43      1606
           8       0.71      0.69      0.70      3898

    accuracy                           0.43     11877
   macro avg       0.36      0.42      0.33     11877
weighted avg       0.48      0.43      0.43     11877


Confusion matrix:
[[ 189  143  121  154  140   85  209  200]
 [ 124  228  157  150  194   74  189  194]
 [   7   12   99   45   25    8    5    2]
 [   7    5   34  193    0   15    8   24]
 [  49   62  137   49  548   79   90   72]
 [ 129  149  229  567  141  327  414  

In [44]:
evaluate_model(
    best_tree,
    X_train,
    y_train
)

Accuracy: 0.4529
Quadratic Kappa: 0.4666
AUC: 0.8042

Classification report:
              precision    recall  f1-score   support

           1       0.36      0.17      0.23      4966
           2       0.39      0.22      0.28      5242
           3       0.13      0.59      0.22       810
           4       0.12      0.76      0.21      1142
           5       0.55      0.51      0.53      4346
           6       0.44      0.16      0.24      8986
           7       0.38      0.51      0.44      6421
           8       0.73      0.71      0.72     15591

    accuracy                           0.45     47504
   macro avg       0.39      0.46      0.36     47504
weighted avg       0.51      0.45      0.45     47504


Confusion matrix:
[[  868   640   555   597   481   320   764   741]
 [  494  1162   549   524   724   352   753   684]
 [   17    15   481   175    61    22    19    20]
 [    7    11   106   871     0    29    42    76]
 [  142   259   627   198  2232   275   350   263

Klasifikaciono stablo ostvarilo je Accuracy vrednost 0.4274, Quadratic Kappa vrednost 0.4252 i AUC vrednost 0.7855. Nije uočena značajna razlika između performansi na trening i validacionom skupu, što ukazuje da nije došlo do izraženog overfittinga. Kao i kod logističke regresije, model pokazuje poteškoće pri klasifikaciji manje zastupljenih klasa, dok je klasa 8, kao najzastupljenija, najuspešnije prepoznata. U poređenju sa logističkom regresijom, stablo je ostvarilo slabije rezultate, što je zanimljivo imajući u vidu da logistička regresija generalno pokazuje veću osetljivost na redundantne i međusobno povezane atribute. S obzirom da je u prvom krugu zadržan veliki broj atributa bez detaljne analize dupliranja informacija, ostaje otvoreno pitanje da li prisustvo redundantnih atributa utiče na performanse modela i predstavlja potencijalni pravac za dalju analizu.

### Random Forest

In [45]:
rf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_estimators=100
    ))
])

In [46]:
rf_pipeline.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [47]:
evaluate_model(
    rf_pipeline,
    X_val,
    y_val
)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.5256
Quadratic Kappa: 0.4689
AUC: 0.8259

Classification report:
              precision    recall  f1-score   support

           1       0.44      0.19      0.26      1241
           2       0.43      0.23      0.30      1310
           3       0.50      0.01      0.03       203
           4       0.47      0.03      0.05       286
           5       0.54      0.53      0.53      1086
           6       0.40      0.50      0.45      2247
           7       0.48      0.30      0.37      1606
           8       0.61      0.90      0.73      3898

    accuracy                           0.53     11877
   macro avg       0.48      0.34      0.34     11877
weighted avg       0.50      0.53      0.49     11877


Confusion matrix:
[[ 234  180    0    1  125  295   86  320]
 [ 109  307    0    1  172  317   85  319]
 [  10   19    3    0   66   75    6   24]
 [   9    4    1    8    0  117   11  136]
 [  31   92    2    0  575  215   52  119]
 [  91   85    0    3  110 1129  180  

In [48]:
param_grid_rf = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20],
    "model__min_samples_leaf": [20, 50],
    "model__max_features": ["sqrt"]
}

grid_rf = GridSearchCV(
    rf_pipeline,
    param_grid_rf,
    cv=3,
    verbose=2,
    n_jobs=1,
    scoring= make_scorer(
        cohen_kappa_score,
        weights="quadratic"
    )

)

grid_rf.fit(X_train, y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 33, 50, 59] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=20, model__n_estimators=100; total time=  17.7s


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 5, 6, 18, 36, 40, 43, 55] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=20, model__n_estimators=100; total time=  17.6s


...

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 5, 6, 18, 36, 40, 43, 55] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=50, model__n_estimators=200; total time=  38.1s


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 37] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=50, model__n_estimators=200; total time=  40.0s


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [10, 20], 'model__max_features': ['sqrt'], 'model__min_samples_leaf': [20, 50], 'model__n_estimators': [100, 200]}"
,scoring,make_scorer(c...hts=quadratic)
,n_jobs,1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehot', ...)]"


In [49]:
print(grid_rf.best_params_)
print(grid_rf.best_score_)

{'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 20, 'model__n_estimators': 100}
0.48377293641560587


In [50]:
best_rf = grid_rf.best_estimator_

evaluate_model(best_rf, X_val, y_val)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.454
Quadratic Kappa: 0.4663
AUC: 0.8138

Classification report:
              precision    recall  f1-score   support

           1       0.32      0.27      0.29      1241
           2       0.35      0.20      0.26      1310
           3       0.14      0.39      0.20       203
           4       0.13      0.64      0.21       286
           5       0.43      0.57      0.49      1086
           6       0.41      0.15      0.22      2247
           7       0.41      0.44      0.42      1606
           8       0.70      0.74      0.72      3898

    accuracy                           0.45     11877
   macro avg       0.36      0.42      0.35     11877
weighted avg       0.48      0.45      0.45     11877


Confusion matrix:
[[ 329  169   61  116  162   64  137  203]
 [ 175  263  105  101  256   62  157  191]
 [  13   11   79   47   32    8   11    2]
 [  16    2   24  184    0   21   14   25]
 [  42   99  103   29  617   40   83   73]
 [ 250  148  150  415  279  334  327  3

Random Forest model ostvario je najvece vrednosti Accuracy i AUC metrike u odnosu na prethodne modele, ali Quadratic Kappa nije znacajno unapredjena. Nije uocena znacajna razlika izmedju trening i validacionog skupa, sto ukazuje da nije doslo do izrazenog overfittinga. Model i dalje pokazuje izrazene poteskoce pri klasifikaciji manje zastupljenih klasa, dok klasa 8 ostvaruje najbolje rezultate. 

### XGBoost

In [51]:
y_train_xgb = y_train - 1

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_xgb
)

In [52]:
xgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=8,
        eval_metric="mlogloss",
        random_state=42
    ))
])


In [53]:
param_grid_xgb = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

In [54]:
grid_xgb = GridSearchCV(
    xgb_pipeline,
    param_grid_xgb,
    cv=3,
    scoring=make_scorer(
        cohen_kappa_score,
        weights="quadratic"
    ),
    verbose=2,
    n_jobs=1
)

In [55]:
grid_xgb.fit(
    X_train,
    y_train_xgb,
    model__sample_weight=sample_weights
)

Fitting 3 folds for each of 32 candidates, totalling 96 fits


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 33, 50, 59] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=3, model__n_estimators=100, model__subsample=0.8; total time=  13.7s


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 5, 6, 18, 36, 40, 43, 55] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__colsample_bytree=0.8, model__learning_rate=0.05, model__max_depth=3, model__n_estimators=100, model__subsample=0.8; total time=  13.6s


...

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 5, 6, 18, 36, 40, 43, 55] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__colsample_bytree=1.0, model__learning_rate=0.1, model__max_depth=5, model__n_estimators=200, model__subsample=1.0; total time=  13.5s


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 37] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV] END model__colsample_bytree=1.0, model__learning_rate=0.1, model__max_depth=5, model__n_estimators=200, model__subsample=1.0; total time=  13.5s


,estimator,"Pipeline(step...ass=8, ...))])"
,param_grid,"{'model__colsample_bytree': [0.8, 1.0], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [3, 5], 'model__n_estimators': [100, 200], ...}"
,scoring,make_scorer(c...hts=quadratic)
,n_jobs,1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('onehot', ...)]"


In [58]:
print(grid_xgb.best_params_)
print(grid_xgb.best_score_)

best_xgb = grid_xgb.best_estimator_




{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 200, 'model__subsample': 0.8}
0.5347036425434293


In [59]:
def evaluate_xgb(model, X, y):
    y_pred = model.predict(X) + 1
    y_prob = model.predict_proba(X)
    print(
        "Accuracy:",
        accuracy_score(
            y,
            y_pred
        )
    )

    print(
        "Quadratic Kappa:",
        cohen_kappa_score(
            y,
            y_pred,
            weights="quadratic"
        )
    )

    print(
        "AUC:",
        roc_auc_score(
            y,
            y_prob,
            multi_class="ovr",
            average="weighted"
        )
    )

    print(
        classification_report(
            y,
            y_pred
        )
    )
    cm = confusion_matrix(
        y,
        y_pred
    )
    print(cm)


evaluate_xgb(best_xgb, X_val, y_val)


C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.4954112991496169
Quadratic Kappa: 0.5202529553832582
AUC: 0.8359740739603603
              precision    recall  f1-score   support

           1       0.39      0.31      0.34      1241
           2       0.43      0.28      0.34      1310
           3       0.15      0.38      0.21       203
           4       0.15      0.67      0.24       286
           5       0.53      0.54      0.53      1086
           6       0.48      0.27      0.35      2247
           7       0.43      0.52      0.47      1606
           8       0.73      0.73      0.73      3898

    accuracy                           0.50     11877
   macro avg       0.41      0.46      0.40     11877
weighted avg       0.53      0.50      0.50     11877

[[ 382  188   55   93  116  108  134  165]
 [ 183  371   90   92  170  105  138  161]
 [  13   12   78   50   30   11    7    2]
 [   7    4   25  193    0   21   11   25]
 [  42   91  102   37  582   82   87   63]
 [ 190  132  125  376  162  608  366  288]
 [

In [61]:
evaluate_xgb(best_xgb, X_train, y_train)

Accuracy: 0.5836771640282924
Quadratic Kappa: 0.6047067928936709
AUC: 0.8881211544183183
              precision    recall  f1-score   support

           1       0.59      0.45      0.51      4966
           2       0.62      0.43      0.51      5242
           3       0.33      0.88      0.48       810
           4       0.21      0.89      0.33      1142
           5       0.65      0.64      0.64      4346
           6       0.58      0.34      0.43      8986
           7       0.48      0.58      0.52      6421
           8       0.76      0.76      0.76     15591

    accuracy                           0.58     47504
   macro avg       0.53      0.62      0.52     47504
weighted avg       0.63      0.58      0.59     47504

[[ 2251   449   230   295   350   335   502   554]
 [  419  2264   244   330   521   383   539   542]
 [    1     3   713    55    11     8     5    14]
 [    1     0    21  1016     0    19    25    60]
 [  144   229   312   112  2761   264   295   229]
 [  5

XGBoost model ostvario je najbolju Quadratic Kappa vrednost u odnosu na prethodno testirane modele, uz Accuracy od 0.495 i AUC od 0.836. Uocena je manja razlika izmedju performansi na trening i validacionom skupu, sto ukazuje na blago prisustvo overfittinga, ali bez znacajnog narusavanja sposobnosti generalizacije. Kao i kod prethodnih modela, najbolje rezultate ostvaruje klasa 8, dok manje zastupljene klase i dalje predstavljaju izazov.

### Potencijalni problemi i dodatno sredjivanje podataka  

- Pristrastnost ka vecinskoj klasi(nebalansiranost klasa)
- Potencijalni sumovi, jaka imputacija NAn

In [64]:
best_xgb.named_steps

{'preprocess': ColumnTransformer(remainder='passthrough',
                   transformers=[('onehot',
                                  OneHotEncoder(drop='first',
                                                handle_unknown='ignore'),
                                  ['Product_Info_1', 'Product_Info_2',
                                   'Product_Info_3', 'Product_Info_5',
                                   'Product_Info_6', 'Product_Info_7',
                                   'Employment_Info_2', 'Employment_Info_3',
                                   'Employment_Info_5', 'InsuredInfo_1',
                                   'InsuredInfo_2', 'InsuredInfo_3',
                                   'InsuredInfo_4', 'InsuredInfo_5',
                                   'InsuredInfo_6', 'InsuredInfo_7',
                                   'Insurance_History_1', 'Insurance_History_2',
                                   'Insurance_History_3', 'Insurance_History_4',
                              

In [74]:
def grouped_feature_importance(best_model):

    feature_names = best_model.named_steps["preprocess"].get_feature_names_out()

    importances = best_model.named_steps["model"].feature_importances_

    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })

    importance["original_feature"] = importance["feature"]

    
    mask = importance["feature"].str.startswith("onehot__")

    importance.loc[mask, "original_feature"] = (
        importance.loc[mask, "feature"]
        .str.replace("onehot__", "", regex=False)
        .str.extract(r"(^.*?)(?:_[^_]*)?$")[0]
    )

    
    importance.loc[~mask, "original_feature"] = (
        importance.loc[~mask, "feature"]
        .str.replace("remainder__", "", regex=False)
    )

    grouped_importance = (
        importance
        .groupby("original_feature")["importance"]
        .mean()
        .sort_values(ascending=False)
    )

    return grouped_importance

In [75]:
print(grouped_feature_importance(best_xgb).head(20))

print(grouped_feature_importance(best_rf).head(20))

print(grouped_feature_importance(best_tree).head(20))

original_feature
Medical_History_4      0.051215
Medical_Keyword_3      0.019859
BMI                    0.015641
Product_Info_4         0.012506
Medical_Keyword_15     0.011310
Medical_History_23     0.010144
Medical_Keyword_23     0.009813
Medical_History_30     0.007983
InsuredInfo_6          0.007582
Insurance_History_2    0.007352
InsuredInfo_7          0.006841
InsuredInfo_5          0.006693
Ins_Age                0.006390
Employment_Info_3      0.006338
Medical_History_40     0.005829
Insurance_History_8    0.005650
InsuredInfo_1          0.005635
Medical_History_33     0.005600
InsuredInfo_2          0.005318
Product_Info_5         0.005313
Name: importance, dtype: float32
original_feature
Medical_History_4      0.146758
BMI                    0.125253
Wt                     0.091084
Product_Info_4         0.064083
Ins_Age                0.042796
Employment_Info_1      0.027202
Ht                     0.023552
Family_Hist_4          0.023418
Employment_Info_6      0.023014
Medic

In [76]:
def grouped_logreg_importance(best_model):

    feature_names = best_model.named_steps["preprocess"].get_feature_names_out()

    importances = np.abs(
        best_model.named_steps["model"].coef_
    ).mean(axis=0)

    importance = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })

    importance["original_feature"] = importance["feature"]

    mask = importance["feature"].str.startswith("onehot__")

    importance.loc[mask, "original_feature"] = (
        importance.loc[mask, "feature"]
        .str.replace("onehot__", "", regex=False)
        .str.extract(r"(^.*?)(?:_[^_]*)?$")[0]
    )

    importance.loc[~mask, "original_feature"] = (
        importance.loc[~mask, "feature"]
        .str.replace("remainder__", "", regex=False)
    )

    grouped_importance = (
        importance
        .groupby("original_feature")["importance"]
        .mean()
        .sort_values(ascending=False)
    )

    return grouped_importance
grouped_logreg_importance(best_log_reg).head(20)

original_feature
BMI                   3.561062
Medical_Keyword_3     1.808762
Wt                    1.614231
Ins_Age               1.208204
Medical_History_4     1.207142
Medical_Keyword_38    0.952735
InsuredInfo_2         0.937447
Ht                    0.790353
Medical_Keyword_41    0.698975
Product_Info_4        0.668550
Medical_History_5     0.587851
Medical_Keyword_15    0.549167
InsuredInfo_5         0.537454
Medical_History_11    0.499139
Medical_History_7     0.492386
Product_Info_2        0.468027
Medical_Keyword_35    0.466020
Family_Hist_5         0.463603
Medical_History_35    0.456236
Medical_Keyword_45    0.453951
Name: importance, dtype: float64

### Testiranje atributa kod kojih je amputiran veliki broj Nan(>50)

 Od znacaja je to da atributi koji su imali veliki procenat missing vrednosti, pojavljuju se kao vazni, a to moze prouzrokovati sledece:  
- stvarno su vazni za izlaz  
- podaci su vestacki imputirani, laznim obrascima, sto moze dovesti do suma 

In [77]:
train_eval, val_eval = get_start_data()

test_cols = [
    "Family_Hist_5",
    "Family_Hist_3"
]

train_eval = train_eval.drop(
    columns=test_cols
)

val_eval = val_eval.drop(
    columns=test_cols
)
X_train, y_train = train_eval.drop(columns="Response"), train_eval["Response"]
X_val, y_val = val_eval.drop(columns="Response"), val_eval["Response"]

In [78]:
cat_cols_exp = [
    col for col in cat_cols
    if col not in test_cols
]

discrete_cols_exp = [
    col for col in discrete_cols
    if col not in test_cols
]

cols_to_encode_exp = cat_cols_exp + discrete_cols_exp

In [79]:
preprocessor_exp = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", drop="first"),
            cols_to_encode_exp
        )
    ],
    remainder="passthrough"
)

In [80]:
log_reg_pipeline_exp = Pipeline([
    ("preprocess", preprocessor_exp),
    ("model", LogisticRegression(
        max_iter=300,
        C=1,
        penalty='l2',
        
        random_state=42,
        class_weight="balanced"
    ))
])

In [81]:
log_reg_pipeline_exp.fit(X_train, y_train)

evaluate_model(log_reg_pipeline_exp, X_val, y_val)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.4291
Quadratic Kappa: 0.4951
AUC: 0.7925

Classification report:
              precision    recall  f1-score   support

           1       0.34      0.33      0.34      1241
           2       0.32      0.24      0.27      1310
           3       0.09      0.42      0.15       203
           4       0.12      0.59      0.21       286
           5       0.35      0.40      0.38      1086
           6       0.43      0.16      0.23      2247
           7       0.38      0.38      0.38      1606
           8       0.72      0.70      0.71      3898

    accuracy                           0.43     11877
   macro avg       0.35      0.40      0.33     11877
weighted avg       0.48      0.43      0.44     11877


Confusion matrix:
[[ 406  190   97   99  116   78  106  149]
 [ 228  312  153   95  232   63  100  127]
 [   9   17   86   54   21    7    2    7]
 [   8    2   57  169    0   20    4   26]
 [  79  133  148   40  438   50  104   94]
 [ 229  189  257  386  238  363  297  

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Izbacivanje Family_Hist_5 i Family_Hist_3 nije dovelo do poboljsanja performansi, vec su rezultati za nijansu losiji. To ukazuje da ovi atributi, uprkos velikom procentu nedostajucih vrednosti, verovatno nose odredjeni koristan signal za model. Zbog toga ih u ovoj fazi ne treba izbacivati.

### Testiranje atributa sa niskim feature importance

In [83]:
print(grouped_feature_importance(best_xgb).tail(30))
print(grouped_feature_importance(best_rf).tail(30))

print(grouped_feature_importance(best_tree).tail(30))
print(grouped_logreg_importance(best_log_reg).tail(30))

original_feature
Medical_History_22     0.002677
Medical_Keyword_6      0.002658
Medical_Keyword_13     0.002654
Medical_Keyword_22     0.002607
Medical_History_36     0.002590
Product_Info_6         0.002568
InsuredInfo_4          0.002558
Insurance_History_3    0.002543
Medical_History_14     0.002503
Medical_History_20     0.002470
Medical_History_19     0.002465
Medical_History_7      0.002403
Medical_History_27     0.002378
Medical_Keyword_33     0.002328
Medical_Keyword_40     0.002327
Medical_History_6      0.002254
Medical_Keyword_21     0.002239
Medical_History_31     0.002143
Medical_History_35     0.002091
Medical_History_41     0.002007
Product_Info_3         0.001820
Medical_Keyword_27     0.001737
Employment_Info_2      0.001671
Medical_History_16     0.001544
Medical_History_21     0.001523
Medical_History_29     0.001509
Medical_History_34     0.001440
Medical_History_37     0.001372
Product_Info_7         0.001320
Employment_Info_5      0.000000
Name: importance, dtype

In [84]:
train_eval, val_eval = get_start_data()

test_cols = [
      "Employment_Info_5",
    "Medical_History_35",
    "Medical_History_31",
    "Medical_Keyword_18",
    "Medical_Keyword_13"
]

train_eval = train_eval.drop(
    columns=test_cols
)

val_eval = val_eval.drop(
    columns=test_cols
)
X_train, y_train = train_eval.drop(columns="Response"), train_eval["Response"]
X_val, y_val = val_eval.drop(columns="Response"), val_eval["Response"]

In [85]:
cat_cols_exp = [
    col for col in cat_cols
    if col not in test_cols
]

discrete_cols_exp = [
    col for col in discrete_cols
    if col not in test_cols
]

cols_to_encode_exp = cat_cols_exp + discrete_cols_exp

In [86]:
preprocessor_exp = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", drop="first"),
            cols_to_encode_exp
        )
    ],
    remainder="passthrough"
)

In [87]:
log_reg_pipeline_exp = Pipeline([
    ("preprocess", preprocessor_exp),
    ("model", LogisticRegression(
        max_iter=300,
        C=1,
        penalty='l2',
        
        random_state=42,
        class_weight="balanced"
    ))
])

In [88]:
log_reg_pipeline_exp.fit(X_train, y_train)

evaluate_model(log_reg_pipeline_exp, X_val, y_val)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 16, 57] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.4318
Quadratic Kappa: 0.4952
AUC: 0.7929

Classification report:
              precision    recall  f1-score   support

           1       0.34      0.32      0.33      1241
           2       0.32      0.23      0.27      1310
           3       0.10      0.43      0.16       203
           4       0.12      0.59      0.20       286
           5       0.35      0.41      0.38      1086
           6       0.42      0.17      0.24      2247
           7       0.39      0.38      0.39      1606
           8       0.72      0.70      0.71      3898

    accuracy                           0.43     11877
   macro avg       0.35      0.40      0.33     11877
weighted avg       0.48      0.43      0.44     11877


Confusion matrix:
[[ 402  184   99   97  122   84  102  151]
 [ 230  307  151   96  232   64  100  130]
 [   9   16   88   53   19    9    2    7]
 [   9    2   58  168    0   21    3   25]
 [  79  131  147   38  440   56   99   96]
 [ 232  189  246  385  242  372  293  

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 16, 57] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Dodatni eksperimenti uklanjanja konfliktnih i atributa sa veoma malom vaznoscu nisu doveli do znacajnog poboljsanja performansi. To ukazuje da model nije posebno osetljiv na pojedinacne atribute i sugerise da glavni izazov verovatno potice od same strukture problema, odnosno preklapanja klasa i nebalansiranosti.

### Analiza balansiranja klasa

In [91]:
train_eval, val_eval = get_start_data()
X_train, y_train = train_eval.drop(columns="Response"), train_eval["Response"]
X_val, y_val = val_eval.drop(columns="Response"), val_eval["Response"]

In [92]:
log_reg_pipeline_smote = Pipeline([
    ("preprocess", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=300, C=1,
    penalty='l2',
     random_state=42))
])

In [94]:
log_reg_pipeline_smote.fit(
    X_train,
    y_train
)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,steps,"[('preprocess', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [96]:
evaluate_model(
    log_reg_pipeline_smote,
    X_val,
    y_val
)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.4316
Quadratic Kappa: 0.4927
AUC: 0.791

Classification report:
              precision    recall  f1-score   support

           1       0.34      0.36      0.35      1241
           2       0.31      0.26      0.28      1310
           3       0.11      0.45      0.18       203
           4       0.13      0.53      0.21       286
           5       0.36      0.39      0.38      1086
           6       0.40      0.18      0.25      2247
           7       0.38      0.39      0.38      1606
           8       0.72      0.68      0.70      3898

    accuracy                           0.43     11877
   macro avg       0.34      0.41      0.34     11877
weighted avg       0.47      0.43      0.44     11877


Confusion matrix:
[[ 442  186   83   69  108  102  107  144]
 [ 251  337  131   73  219   77   87  135]
 [  12   18   92   44   19    9    4    5]
 [   9    6   58  153    0   23    6   31]
 [  89  155  126   35  427   63  104   87]
 [ 262  222  214  336  219  411  303  2

In [98]:
smote_rf = Pipeline([
    ("preprocess", preprocessor),

    ("smote", SMOTE(
        random_state=42
    )),

    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

smote_rf.fit(
    X_train,
    y_train
)

evaluate_model(
    smote_rf,
    X_val,
    y_val
)

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 17, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Accuracy: 0.5177
Quadratic Kappa: 0.4882
AUC: 0.8235

Classification report:
              precision    recall  f1-score   support

           1       0.36      0.24      0.29      1241
           2       0.38      0.24      0.29      1310
           3       0.22      0.10      0.14       203
           4       0.27      0.27      0.27       286
           5       0.50      0.54      0.52      1086
           6       0.42      0.43      0.42      2247
           7       0.45      0.37      0.41      1606
           8       0.65      0.84      0.74      3898

    accuracy                           0.52     11877
   macro avg       0.41      0.38      0.38     11877
weighted avg       0.49      0.52      0.50     11877


Confusion matrix:
[[ 299  187    6   18  133  229  102  267]
 [ 175  317   13   21  185  228  106  265]
 [  15   23   20   13   53   52   11   16]
 [  11    4    7   76    0   86   17   85]
 [  56  117   17    4  586  147   62   97]
 [ 161  141   21   70  162  962  246  

SMOTE kreira sintetičke uzorke interpolacijom između postojećih instanci manjinske klase. U ovom radu skup podataka sadrži veliki broj atributa, uključujući veliki broj binarnih indikatora (Medical_Keyword) i kompleksne obrasce među promenljivama. U takvom prostoru visoke dimenzionalnosti sintetički generisani uzorci mogu biti manje reprezentativni i mogu dovesti do unošenja dodatnog šuma.

Analiza je sprovedena koriscenjem algoritma logisticke regresije, kao i random forest-a, medjutim SMOTE nije uspeo da da bolje rezultate. 

Iako razlike između modela nisu bile velike, XGBoost je ostvario najbolje rezultate prema Quadratic Kappa metrici i AUC vrednosti, zbog čega je izabran kao finalni model.

## Finalni model

In [103]:
train_full = train.copy()
test_full = test.copy()

# 1. imputacija numerickih
numeric_with_missing = [
    col for col in continuous_cols
    if train_full[col].isnull().sum() > 0
]

num_imputer = SimpleImputer(strategy="median")

train_full[numeric_with_missing] = num_imputer.fit_transform(
    train_full[numeric_with_missing]
)

test_full[numeric_with_missing] = num_imputer.transform(
    test_full[numeric_with_missing]
)


# 2. imputacija diskretnih/kategorijskih sa missing
discrete_with_missing = [
    col for col in discrete_cols
    if train_full[col].isnull().sum() > 0
]

cat_imputer = SimpleImputer(strategy="most_frequent")

train_full[discrete_with_missing] = cat_imputer.fit_transform(
    train_full[discrete_with_missing]
)

test_full[discrete_with_missing] = cat_imputer.transform(
    test_full[discrete_with_missing]
)


# 3. rare kategorije za Medical_History_2
freq = train_full["Medical_History_2"].value_counts(normalize=True) * 100
rare = freq[freq < 0.5].index

train_full["Medical_History_2"] = train_full["Medical_History_2"].apply(
    lambda x: "Other" if x in rare else x
)

test_full["Medical_History_2"] = test_full["Medical_History_2"].apply(
    lambda x: "Other" if x in rare else x
)

train_full["Medical_History_2"] = train_full["Medical_History_2"].astype(str).astype("category")
test_full["Medical_History_2"] = test_full["Medical_History_2"].astype(str).astype("category")


# 4. rare kategorije za Medical_History_1
freq = train_full["Medical_History_1"].value_counts(normalize=True) * 100
rare = freq[freq < 0.5].index

train_full["Medical_History_1"] = train_full["Medical_History_1"].apply(
    lambda x: "Other" if x in rare else x
)

test_full["Medical_History_1"] = test_full["Medical_History_1"].apply(
    lambda x: "Other" if x in rare else x
)

train_full["Medical_History_1"] = train_full["Medical_History_1"].astype(str).astype("category")
test_full["Medical_History_1"] = test_full["Medical_History_1"].astype(str).astype("category")

In [120]:
X_full = train_full.drop(columns="Response")
y_full = train_full["Response"]

sample_weights_full = compute_sample_weight(
    class_weight="balanced",
    y=y_full - 1
)

final_xgb = Pipeline([
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=grid_xgb.best_params_["model__n_estimators"],
        max_depth=grid_xgb.best_params_["model__max_depth"],
        learning_rate=grid_xgb.best_params_["model__learning_rate"],
        subsample=grid_xgb.best_params_["model__subsample"],
        colsample_bytree=grid_xgb.best_params_["model__colsample_bytree"],
        objective="multi:softprob",
        num_class=8,
        eval_metric="mlogloss",
        random_state=42
    ))
])


In [121]:
final_xgb.fit(
    X_full,
    y_full - 1,
    model__sample_weight=sample_weights_full
)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [122]:
pred = final_xgb.predict(test_full) + 1

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 24, 51, 56, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [116]:
test_id = pd.read_csv("../rezultati/test_id.csv")
test_id = test_id["Id"]

In [117]:
pred = final_xgb.predict(test_full) + 1

C:\Users\PC\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2, 6, 24, 51, 56, 60] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [118]:
submission = pd.DataFrame({
    "Id": test_id,
    "Response": pred
})

submission.head()

,Id,Response
0,1,1
1,3,4
2,4,6
3,9,7
4,12,4


In [119]:
submission.to_csv("../rezultati/submission.csv", index=False)